# 🧱 Part 02: BPE Tokenizer: Growing a Vocabulary From Statistics

> **Previous context**: Part 01 showed why raw text must become token IDs, and why character-level and word-level tokenization both have trade-offs.
> **Goal for this part**: Implement Byte Pair Encoding (BPE) by counting frequent neighboring pairs and merging them step by step.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why BPE exists

BPE solves the tension between tiny character vocabularies and huge word vocabularies. It starts with small pieces and lets frequent patterns become larger tokens.

## 1. Core action

The whole algorithm repeats one move: count adjacent token pairs, find the most frequent pair, merge it into a new token, and record that merge rule.

## 2. Manual example

If ('l', 'o') appears most often, BPE creates 'lo'. Later ('lo', 'w') might become 'low'. The vocabulary is not magic; it grows from repeated statistics.

## 3. Industrial difference

Real tokenizers use byte-level handling, normalization rules, special tokens, and large corpora. This notebook keeps the algorithm small so every merge is visible.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] BPE begins from small units and repeatedly merges frequent adjacent pairs.
- [ ] Merge rules define how encoding works later.
- [ ] Subword tokenization avoids most OOV failures while keeping sequences shorter than character-level tokenization.

Next, continue through the code cells for the Foundation part and inspect the printed observations.


## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Problem|
|:---|:---|:---|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|OOV (Out Of Vocabulary) demo|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

Read the values printed above and connect them to the concept in this cell.

In [1]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print('Read the values printed above and connect them to the concept in this cell.')
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

Read the values printed above and connect them to the concept in this cell.  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends


## 2. training BPE
### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [2]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

def text_to_tokens(text):
    'Read the values printed above and connect them to the concept in this cell.'
    return list(text)

# Teaching note: follow this line to see the main step.
sentence = corpus[0]
chars = text_to_tokens(sentence)

print(f"Original text:   '{sentence}'")
print(f"Read the values printed above and connect them to the concept in this cell.{chars}")
print(f"Read the values printed above and connect them to the concept in this cell.{len(chars)}Read the values printed above and connect them to the concept in this cell.")

# Teaching note: follow this line to see the main step.
corpus_tokens = [text_to_tokens(s) for s in corpus]
print(f"Read the values printed above and connect them to the concept in this cell.{[len(t) for t in corpus_tokens]}")

Original text: 'the cat sat on the mat'Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↑──↑   ↑──↑   ↑──↑   ↑──↑   ↑──↑
 ('t','h') ('h','e') ('e',' ') (' ','c') ('c','a') ('a','t')
```

Read the values printed above and connect them to the concept in this cell.

In [3]:
from collections import defaultdict

def count_pairs(token_lists):
    'Read the values printed above and connect them to the concept in this cell.'
    pairs = defaultdict(int)
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
    return dict(pairs)

# Teaching note: follow this line to see the main step.
initial_pairs = count_pairs(corpus_tokens)

print('Read the values printed above and connect them to the concept in this cell.')
# Teaching note: follow this line to see the main step.
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print(f"  {pair}: {count}Read the values printed above and connect them to the concept in this cell.")

print(f"Read the values printed above and connect them to the concept in this cell.{max(initial_pairs, key=initial_pairs.get)}")
print(f"Read the values printed above and connect them to the concept in this cell.{initial_pairs[max(initial_pairs, key=initial_pairs.get)]}Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed a

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

In [4]:
def merge_pair(token_lists, pair_to_merge):
    'Read the values printed above and connect them to the concept in this cell.'
    a, b = pair_to_merge
    new_token = a + b  # Teaching note: follow this line to see the main step.
    
    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            # Teaching note: follow this line to see the main step.
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2  # Teaching note: follow this line to see the main step.
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)
    
    return new_token_lists, new_token

# Teaching note: follow this line to see the main step.
best_pair = max(initial_pairs, key=initial_pairs.get)
print(f"Read the values printed above and connect them to the concept in this cell.{best_pair}")

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print(f"new token: '{new_token}'")
print(f"Read the values printed above and connect them to the concept in this cell.")
for i, tokens in enumerate(merged_tokens):
    print(f"  [{i}] {tokens}")

Read the values printed above and connect them to the concept in this cell.new token: 'e '
Read the values printed above and connect them to the concept in this cell.  [0] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'm', 'a', 't']
  [1] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 's', 'a', 't', ' ', 'o', 'n', ' ', 't', 'h', 'e ', 'l', 'o', 'g']
  [2] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'a', 'n', 'd', ' ', 't', 'h', 'e ', 'd', 'o', 'g']
  [3] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'c', 'a', 't']
  [4] ['i', ' ', 'l', 'o', 'v', 'e ', 'm', 'y', ' ', 'd', 'o', 'g']
  [5] ['t', 'h', 'e ', 'c', 'a', 't', ' ', 'i', 's', ' ', 'c', 'u', 't', 'e']
  [6] ['t', 'h', 'e ', 'd', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'p', 'p', 'y']
  [7] ['t', 'h', 'e ', 'm', 'a', 't', ' ', 'i', 's', ' ', 's', 'o', 'f', 't']
  [8] ['t', 'h', 'e ', 'l', 'o', 'g', ' ', 'i', 's', ' ', 'h', 'a', 'r', 'd']
  [9] ['c', 'a', 't', 's', ' ', 'a', 'n', 'd', ' ', 'd', 'o', 'g', 's

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [5]:
class BPETokenizer:
    'Read the values printed above and connect them to the concept in this cell.'
    
    def __init__(self):
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.merge_rules = []
        # Teaching note: follow this line to see the main step.
        self.vocab = set()
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts, num_merges=15, verbose=True):
        'Read the values printed above and connect them to the concept in this cell.'
        # Teaching note: follow this line to see the main step.
        token_lists = [list(text) for text in texts]
        base_vocab = set(c for tokens in token_lists for c in tokens)
        learned_vocab = set(base_vocab)
        
        if verbose:
            print(f"{'='*60}")
            print(f"Read the values printed above and connect them to the concept in this cell.")
            print(f"{'='*60}")
            print(f"Read the values printed above and connect them to the concept in this cell.{len(set(c for t in token_lists for c in t))}")
            print()
        
        for step in range(num_merges):
            # Teaching note: follow this line to see the main step.
            pairs = defaultdict(int)
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pairs[(tokens[i], tokens[i + 1])] += 1
            
            if not pairs:
                break
            
            # Teaching note: follow this line to see the main step.
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            
            # Teaching note: follow this line to see the main step.
            self.merge_rules.append(best_pair)
            
            # Teaching note: follow this line to see the main step.
            a, b = best_pair
            new_token = a + b
            
            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)
            
            token_lists = new_token_lists
            
            learned_vocab.add(new_token)
            
            if verbose:
                print(f"Step {step+1:2d}: merge {best_pair} → '{new_token}Read the values printed above and connect them to the concept in this cell.{best_count}Vocabulary size: {len(learned_vocab)}")
        
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.vocab = learned_vocab
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Training finished！")
            print(f"Read the values printed above and connect them to the concept in this cell.{len(self.merge_rules)}")
            print(f"Vocabulary size: {len(self.vocab)}Read the values printed above and connect them to the concept in this cell.")
            print(f"  - Merge rules: {self.merge_rules}")
            print(f"{'='*60}")
        
        return token_lists


In [6]:
# Teaching note: follow this line to see the main step.
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=True)

Read the values printed above and connect them to the concept in this cell.============================================================
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and conn

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Explanation: the printed values show the main mechanism in this step.

## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.
```
"the cat"
  → ['t','h','e',' ','c','a','t']
Read the values printed above and connect them to the concept in this cell.  → ['t','h','e ','c','a','t']
Read the values printed above and connect them to the concept in this cell.  → ['th','e ','c','a','t']
Read the values printed above and connect them to the concept in this cell.  → ['the ','c','a','t']
Read the values printed above and connect them to the concept in this cell.  → ['the ','c','at']
```

Read the values printed above and connect them to the concept in this cell.

In [7]:
# Teaching note: follow this line to see the main step.
def bpe_encode(self, text):
    'Read the values printed above and connect them to the concept in this cell.'
    # Teaching note: follow this line to see the main step.
    tokens = list(text)
    
    # Teaching note: follow this line to see the main step.
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    # Teaching note: follow this line to see the main step.
    ids = []
    for token in tokens:
        if token in self.stoi:
            ids.append(self.stoi[token])
        else:
            # Teaching note: follow this line to see the main step.
            for ch in token:
                ids.append(self.stoi[ch])
    return ids

# Teaching note: follow this line to see the main step.
BPETokenizer.encode = bpe_encode

print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.

In [8]:
# Teaching note: follow this line to see the main step.
text = "the cat"
ids = bpe.encode(text)
print(f"Original text: '{text}'")
print(f"Token IDs: {ids}")
print(f"Number of tokens: {len(ids)}")

# Teaching note: follow this line to see the main step.
print(f"Explanation: the printed values show the main mechanism in this step.")
for i, tid in enumerate(ids):
    print(f"  ID {tid} → '{bpe.itos[tid]}'")

# Teaching note: follow this line to see the main step.
print(f"Original text: {len(text)}Number of tokens: {len(ids)}Read the values printed above and connect them to the concept in this cell.{len(text)/len(ids):.1f}x")

Original text: 'the cat'Token IDs: [30, 3]
Number of tokens: 2
Explanation: the printed values show the main mechanism in this step.  ID 30 → 'the c'
  ID 3 → 'at'

Number of tokens: 

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [9]:
def bpe_decode(self, ids):
    'Read the values printed above and connect them to the concept in this cell.'
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# Teaching note: follow this line to see the main step.
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "[ok]" if test_text == recovered else "[x]"
    print(f"{status} '{test_text}' → {ids} → '{recovered}'")

✅ 'the cat' → [30, 3] → 'the cat'
✅ 'my dog is happy' → [16, 34, 0, 7, 0, 14, 12, 2, 21, 21, 34] → 'my dog is happy'
✅ 'i love cats' → [13, 0, 15, 19, 33, 9, 5, 3, 23] → 'i love cats'


Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```python
if token in self.stoi:
    ids.append(self.stoi[token])
else:
Read the values printed above and connect them to the concept in this cell.        ids.append(self.stoi[ch])
```

Reason
Read the values printed above and connect them to the concept in this cell.

In [10]:
# Teaching note: follow this line to see the main step.
unseen = "a cat runs fast"
print(f"Read the values printed above and connect them to the concept in this cell.{unseen}'")
print(f"Read the values printed above and connect them to the concept in this cell.")

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"\nToken IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"Read the values printed above and connect them to the concept in this cell.{'Read the values printed above and connect them to the concept in this cell.' if unseen == recovered else '⚠️'}")

# Teaching note: follow this line to see the main step.
print(f"Inspect each token:")
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Token IDs: [2, 0, 5, 4, 22, 32, 17, 24, 10, 2, 23, 27]
Decoded back: 'a cat runs fast'Read the values printed above and connect them to the concept in this cell.
Inspect each token:  'a'  ' '  'c'  'at '  'r'  'u'  'n'  's '  'f'  'a'  's'  't'


## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [11]:
# Teaching note: follow this line to see the main step.
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)
    
    test = "the cat sat on the mat"
    ids = t.encode(test)
    
    print(f"merge={n_merges:2d} | vocabulary={len(t.vocab):2d}Read the values printed above and connect them to the concept in this cell.{len(ids):2d} | tokens: {[t.itos[i] for i in ids]}")

print()
print('Number of tokens: ')

Number of tokens: Number of tokens: Number of tokens: 
Number of tokens: 

## Read the values printed above and connect them to the concept in this cell.
### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```text
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|:---|:---|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|

OOV (Out Of Vocabulary) demo
Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.

In [12]:
# Teaching note: follow this line to see the main step.
try:
    import tiktoken

    real_tokenizer_name = "gpt2"
    real_tokenizer = tiktoken.get_encoding(real_tokenizer_name)
    print(f"a real tokenizer: {real_tokenizer_name}")
    print(f"Vocabulary size: {real_tokenizer.n_vocab}")
except Exception as e:
    real_tokenizer = None
    print('Could not load tiktoken. Install tiktoken to run the real tokenizer demo.')
    print(f"Reason{e}")
    print('Real tokenizer: GPT-2 byte-level BPE')


def show_real_tokenization(text):
    'Real tokenizer: GPT-2 byte-level BPE'
    ids = real_tokenizer.encode(text)
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]

    print(f"Text: {text!r}")
    print(f"Number of tokens: {len(ids)}")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(real_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")
    return ids, tokens


if real_tokenizer is not None:
    show_real_tokenization("the cat sat on the mat")


a real tokenizer: gpt2Vocabulary size: 50257Text: 'the cat sat on the mat'Number of tokens:   00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]


### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [13]:
def try_mini_bpe(text):
    'Reason'
    try:
        ids = bpe.encode(text)
        tokens = [bpe.itos[i] for i in ids]
        return ids, tokens, None
    except Exception as e:
        return None, None, e


compare_texts = [
    "the cat sat on the mat",
    " the cat",
    "the  cat",
    "327 + 1 = 328",
    "def f():\n    return x",
    'Read the values printed above and connect them to the concept in this cell.',
]

if real_tokenizer is not None:
    for text in compare_texts:
        print("=" * 68)
        print(f"Text: {text!r}")

        mini_ids, mini_tokens, error = try_mini_bpe(text)
        if error is None:
            print(f"mini BPE: {len(mini_ids)} tokens")
            print(f"  tokens: {mini_tokens}")
            print(f"  ids:    {mini_ids}")
        else:
            print('Read the values printed above and connect them to the concept in this cell.')
            print(f"Reason{error!r}")

        real_ids = real_tokenizer.encode(text)
        real_tokens = [real_tokenizer.decode([tok_id]) for tok_id in real_ids]
        real_tokens = [t.replace("\n", "\\n") for t in real_tokens]
        print(f"Read the values printed above and connect them to the concept in this cell.{len(real_ids)} tokens")
        print(f"  tokens: {real_tokens}")
        print(f"  ids:    {real_ids}")

    print("=" * 68)
    print('Real tokenizer: GPT-2 byte-level BPE')
    print('Read the values printed above and connect them to the concept in this cell.')


Text: 'the cat sat on the mat'mini BPE: 6 tokens
  tokens: ['the cat ', 'sat o', 'n', ' the ', 'm', 'at']
  ids:    [31, 26, 17, 1, 16, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: ['the', ' cat', ' sat', ' on', ' the', ' mat']
  ids:    [1169, 3797, 3332, 319, 262, 2603]
Text: ' the cat'mini BPE: 3 tokens
  tokens: [' ', 'the c', 'at']
  ids:    [0, 30, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: [' the', ' cat']
  ids:    [262, 3797]
Text: 'the cat'mini BPE: 4 tokens
  tokens: ['the ', ' ', 'c', 'at']
  ids:    [29, 0, 5, 3]
Read the values printed above and connect them to the concept in this cell.  tokens: ['the', ' ', ' cat']
  ids:    [1169, 220, 3797]
Text: '327 + 1 = 328'Read the values printed above and connect them to the concept in this cell.ReasonRead the values printed above and connect them to the concept in this cell.  tokens: ['327', ' +', ' 1', ' =', ' 328']
  ids:    [34159, 1343, 3

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Explanation: the printed values show the main mechanism in this step.Read the values printed above and connect them to the concept in this cell.

In [14]:
special_cases = [
    ('Read the values printed above and connect them to the concept in this cell.', "327 + 1 = 328"),
    ('Read the values printed above and connect them to the concept in this cell.', "cat"),
    ('Read the values printed above and connect them to the concept in this cell.', " cat"),
    ('Read the values printed above and connect them to the concept in this cell.', "the    cat"),
    ('Read the values printed above and connect them to the concept in this cell.', "def f():\n    return x"),
    ('Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.'),
]

if real_tokenizer is not None:
    for label, text in special_cases:
        print("=" * 68)
        print(f"Read the values printed above and connect them to the concept in this cell.{label}")
        show_real_tokenization(text)

    print("=" * 68)
    print("Special token: <|endoftext|>")
    text = "hello<|endoftext|>world"

    try:
        real_tokenizer.encode(text)
    except ValueError as e:
        print('Read the values printed above and connect them to the concept in this cell.')
        print(f"Read the values printed above and connect them to the concept in this cell.{str(e).splitlines()[0]}")

    ids = real_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
    print('Read the values printed above and connect them to the concept in this cell.')
    print(f"  ids:    {ids}")
    print(f"  tokens: {tokens}")
    print('Read the values printed above and connect them to the concept in this cell.')


Read the values printed above and connect them to the concept in this cell.Text: '327 + 1 = 328'Number of tokens:   00 | id=34159  | token='327' | bytes=[51, 50, 55]
  01 | id=1343   | token=' +' | bytes=[32, 43]
  02 | id=352    | token=' 1' | bytes=[32, 49]
  03 | id=796    | token=' =' | bytes=[32, 61]
  04 | id=39093  | token=' 328' | bytes=[32, 51, 50, 56]
Read the values printed above and connect them to the concept in this cell.Text: 'cat'Number of tokens:   00 | id=9246   | token='cat' | bytes=[99, 97, 116]
Read the values printed above and connect them to the concept in this cell.Text: ' cat'Number of tokens:   00 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
Read the values printed above and connect them to the concept in this cell.Text: 'the cat'Number of tokens:   00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=220    | token=' ' | bytes=[32]
  02 | id=220    | token=' ' | bytes=[32]
  03 | id=220    | token=' ' | bytes=[32]
  04 | id=3797   | token=' 

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```text
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
```text
1.9  = 1.90
1.11 = 1.11
So 1.90 > 1.11```

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```text
"1.11" → ["1", ".", "11"]
"1.9"  → ["1", ".", "9"]
```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [ ]:
# Teaching note: follow this line to see the main step.
number_texts = ["1.11", "1.9"]

if real_tokenizer is not None:
    for text in number_texts:
        ids = real_tokenizer.encode(text)
        tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
        print(f"{text!r} -> tokens={tokens}, ids={ids}")

    print()
    print('Read the values printed above and connect them to the concept in this cell.')
    print(f"  1.11 > 1.9  ? {1.11 > 1.9}")
    print(f"  1.90 > 1.11 ? {1.90 > 1.11}")
    print()
    print('Key observation: inspect the values above and connect them to the idea in this cell.')
    print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```text
Read the values printed above and connect them to the concept in this cell.hand-designed control symbols: <BOS>、<EOS>、<PAD>、<think>、</think>```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [15]:
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()

for i, (a, b) in enumerate(bpe.merge_rules):
    arrow = ""
    # Teaching note: follow this line to see the main step.
    merged = a + b
    if merged in ['th', 'the ', 'the c', 'the cat ']:
        arrow = 'Read the values printed above and connect them to the concept in this cell.'
    if a in ['th', 'the ', 'the c'] or b in ['e ', 'c', 'at ']:
        arrow = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' → '{merged}'{arrow}")

print()
print('Read the values printed above and connect them to the concept in this cell.')

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
  Rule  1: 'e' + ' ' → 'e '
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Rule  4: 'a' + 't' → 'at'
  Rule  5: 'o' + 'g' → 'og'
  Rule  6: 'at' + ' ' → 'at '
  Rule  7: 's' + ' ' → 's '
  Rule  8: 'd' + 'og' → 'dog'
  Rule  9: 'i' + 's ' → 'is '
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  Rule 12: ' ' + 'the ' → ' the '
  Rule 13: 'n' + 'd' → 'nd'
Read the values printed above and connect them to the concept in this cell.  Rule 15: 'sat ' + 'o' → 'sat o'

Read the values printed above and connect them to the concept in this cell.

## Summary
Read the values printed above and connect them to the concept in this cell.
- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

## Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.
> Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [ ]:
# Teaching note: follow this line to see the main step.
from collections import defaultdict

tokens = list("banana")
pair_counts = defaultdict(int)

# Teaching note: follow this line to see the main step.
'TODO: replace this placeholder with your code'

assert pair_counts[("a", "n")] == 2, dict(pair_counts)
assert pair_counts[("n", "a")] == 2, dict(pair_counts)
assert pair_counts[("b", "a")] == 1, dict(pair_counts)
print('Exercise passed: you have understood this step.')

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [ ]:
# Teaching note: follow this line to see the main step.
def merge_pair(tokens, pair):
    'Read the values printed above and connect them to the concept in this cell.'
    merged = []
    i = 0
    while i < len(tokens):
        # Teaching note: follow this line to see the main step.
        'TODO: replace this placeholder with your code'
    return merged

result = merge_pair(list("banana"), ("a", "n"))

assert result == ["b", "an", "an", "a"], result
print('Exercise passed: you have understood this step.')

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [ ]:
# Teaching note: follow this line to see the main step.
text = 'Read the values printed above and connect them to the concept in this cell.'

# Teaching note: follow this line to see the main step.
byte_values = 'TODO: replace this placeholder with your code'

assert not isinstance(byte_values, str), 'Please replace the placeholder before running the assertion.'
assert byte_values == [228, 189, 160, 240, 159, 152, 138], byte_values
print('Exercise passed: you have understood this step.')